In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from simulations import simulations as sim

# cerebellar ROIs
df1 = sim.y_sim()
df2 = sim.y_sim(seed = 9, region = 'region2')
df3 = sim.y_sim(seed = 99, region = 'region3')
df = pd.concat([df1, df2, df3], axis=0)

# tract ROIs
r1 = sim.y_sim(seed = 1, region = 'tract1')
r2 = sim.y_sim(seed = 2, region = 'tract2')
df = pd.concat([df, r1, r2], axis=0)

In [2]:
def _predict_roi_df(response_df, predict_df):
   # this will give a dataframe for each response_roi-week, you have vlaues of all weeks for each pred_roi
   predictors_df = predict_df.rename(columns = {'regionname': 'pred_region', 'mean': 'pred_mean', 'Week': 'pred_week'})
   roi_df = response_df.merge(predictors_df, on = 'subj_id', how = 'inner')


   return roi_df


response_df = pd.concat([df1, df2, df3], axis = 0)
predict_df = pd.concat([r1, r2], axis = 0)

roi_df = _predict_roi_df(response_df, predict_df)

In [7]:
models = {}
for pred_region in roi_df.pred_region.unique():
    for response_region in roi_df.regionname.unique():
        region_df = roi_df[(roi_df.pred_region == pred_region) & (roi_df.regionname == response_region)]
        model = smf.mixedlm("mean~C(Week)*C(pred_week)*pred_mean", data = region_df, groups = region_df.subj_id).fit(maxiter = 400)
        models[(pred_region, response_region)] = model

In [9]:
models

{('tract1',
  'region1'): <statsmodels.regression.mixed_linear_model.MixedLMResultsWrapper at 0x7dbe8f14f640>,
 ('tract1',
  'region2'): <statsmodels.regression.mixed_linear_model.MixedLMResultsWrapper at 0x7dbe851a83a0>,
 ('tract1',
  'region3'): <statsmodels.regression.mixed_linear_model.MixedLMResultsWrapper at 0x7dbe84323fa0>,
 ('tract2',
  'region1'): <statsmodels.regression.mixed_linear_model.MixedLMResultsWrapper at 0x7dbe84321330>,
 ('tract2',
  'region2'): <statsmodels.regression.mixed_linear_model.MixedLMResultsWrapper at 0x7dbe84321e70>,
 ('tract2',
  'region3'): <statsmodels.regression.mixed_linear_model.MixedLMResultsWrapper at 0x7dbe843107f0>}